In [ ]:
import os
import gc
import glob
import random
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms

from PIL import Image
from PIL import Image as PILImage
from tqdm.auto import tqdm

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import preprocess_image

from lime import lime_image
import shap

import quantus
from quantus import MaxSensitivity, norm_func, similarity_func

from xai_lib import MemoryManagement as mm
from xai_lib import AttributionEngine as att_engine


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Fixed 224x224 inputs -> let cuDNN pick the fastest conv algorithms
torch.backends.cudnn.benchmark = True

In [ ]:
name = "aveera"

IDRID_DIR = f"/data/{name}/Refined IDRiD"
PREPROCESSED_DIR = f"/data/{name}/preprocessed_idrid"
IDRID_SPLIT_DIRS = [
    os.path.join(IDRID_DIR, "Test", "Test"),
    os.path.join(IDRID_DIR, "Train", "Train"),
]

drive_model_dir = f"/data/{name}/models"
RESULTS_DIR = f"/data/{name}/results"
CHECKPOINT_DIR = f"/data/{name}/results/maxsens_checkpoints"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for split_dir in IDRID_SPLIT_DIRS:
    assert os.path.isdir(os.path.join(split_dir, "Images")), (
        f"Expected an 'Images' folder inside {split_dir} -- check IDRID_DIR."
    )
print(f"Using Refined IDRiD data from '{IDRID_DIR}' (Test + Train splits)")


**Dataset: Refined IDRiD**

In [ ]:
def load_preprocessed_from_disk(save_dir):
    images_out_dir = os.path.join(save_dir, "images")
    masks_out_dir = os.path.join(save_dir, "masks")

    image_paths = sorted(glob.glob(os.path.join(images_out_dir, "*.png")))
    if not image_paths:
        raise FileNotFoundError(f"No preprocessed images found in {images_out_dir}")

    samples = []
    for img_path in image_paths:
        image_id = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(masks_out_dir, f"{image_id}.png")

        img = np.array(Image.open(img_path).convert('RGB'))
        mask = np.array(Image.open(mask_path).convert('L'))
        samples.append((img, mask))
    return samples

In [ ]:
# MaxSensitivity only needs the first N_TEST samples
N_TEST = 81
idrid_subset = load_preprocessed_from_disk(PREPROCESSED_DIR)

print(f"Loaded {len(idrid_subset)} IDRiD images (capped at N_TEST={N_TEST})")

maxsens_samples = idrid_subset[:N_TEST]


In [ ]:
_engine_cache = {}

def get_engine_for(model):
    """Look up the AttributionEngine that was built for this exact model object."""
    eng = _engine_cache.get(id(model))
    if eng is None:
        raise RuntimeError("No AttributionEngine registered for this model.")
    return eng


In [ ]:
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def get_predicted_labels(model, samples, device, transform=test_transform):
    preds = []
    model.eval()
    with torch.no_grad():
        for raw_img_np, _ in samples:
            input_tensor = transform(PILImage.fromarray(raw_img_np)).unsqueeze(0).to(device)
            pred = model(input_tensor).argmax(dim=1).item()
            preds.append(pred)
    return np.array(preds)


In [ ]:
def explain_gradcam(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        gradcam = eng.compute_gradcam(img_tensor.unsqueeze(0).to(eng.device), int(target))
        maps.append(gradcam)
    return np.stack(maps)[:, None, :, :]


def explain_lime(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        raw = eng.unnormalize_to_uint8(img_tensor)
        explanation = eng.compute_lime(raw, int(target))
        heatmap = eng.get_lime_heatmap(explanation, int(target))
        maps.append(heatmap)
        eng.close_lime_explanation(explanation)
        del explanation
    return np.stack(maps)[:, None, :, :]


def explain_shap(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        raw = eng.unnormalize_to_uint8(img_tensor)
        shap_values = eng.compute_shap(raw, target_class=int(target), all_classes=False)
        heatmap, _ = eng.get_shap_heatmap(shap_values, class_position=0)
        maps.append(heatmap)
    return np.stack(maps)[:, None, :, :]


# Kept as an ordered dict-of-one so we can loop "one method at a time" deliberately below
explain_funcs = {
    "GradCAM": explain_gradcam,
    "LIME": explain_lime,
    "SHAP": explain_shap,
}


**Checkpointing**

In [ ]:
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "maxsens_per_image_checkpoint.csv")

CHECKPOINT_COLUMNS = ["model", "method", "image_index", "max_sensitivity"]


def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=CHECKPOINT_COLUMNS)


def append_checkpoint_row(row: dict):
    """Append a single completed (model, method, image_index) result to disk immediately."""
    df_row = pd.DataFrame([row], columns=CHECKPOINT_COLUMNS)
    write_header = not os.path.exists(CHECKPOINT_PATH)
    df_row.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)


def already_done(checkpoint_df, model_name, method, image_index):
    if checkpoint_df.empty:
        return False
    match = checkpoint_df[
        (checkpoint_df["model"] == model_name)
        & (checkpoint_df["method"] == method)
        & (checkpoint_df["image_index"] == image_index)
    ]
    return not match.empty


**Max-sensitivity configuration**

In [ ]:
MAXSENS_KWARGS = dict(
    nr_samples=10,
    lower_bound=0.2,
    norm_numerator=norm_func.fro_norm,
    norm_denominator=norm_func.fro_norm,
    similarity_func=similarity_func.difference,
)

MODEL_NAMES = ["resnet50", "efficientnetb0"]
METHOD_NAMES = ["GradCAM", "LIME", "SHAP"]  


LIME_SAMPLES    = 1000   
SHAP_MAX_EVALS  = 500    
PREDICT_CHUNK_SIZE = 32  


**Running Max-sensitivity**

In [ ]:
checkpoint_df = load_checkpoint()
print(f"Resuming from checkpoint with {len(checkpoint_df)} previously completed (model, method, image) rows.")

for model_name in MODEL_NAMES:
    print(f"\n{'='*60}\n  MaxSensitivity | Model: {model_name}\n{'='*60}")

    # ---- Load ONE model at a time ----
    drive_weight_path = os.path.join(drive_model_dir, f"{model_name}_best_model.pth")
    model = mm.load_model(model_name, drive_weight_path, device)

    engine = att_engine.AttributionEngine(
        model, device,
        lime_samples=LIME_SAMPLES,
        shap_max_evals=SHAP_MAX_EVALS,
        predict_chunk_size=PREDICT_CHUNK_SIZE,
    )
    _engine_cache[id(model)] = engine

    input_tensors = []
    for raw_img_np, _ in maxsens_samples:
        image_np = raw_img_np.astype(np.float32) / 255.0
        t = preprocess_image(image_np, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        input_tensors.append(t.squeeze(0))
    input_batch = torch.stack(input_tensors)
    x_batch_all = input_batch.cpu().numpy()
    del input_tensors, input_batch
    mm.clear_memory()

    y_batch_all = get_predicted_labels(model, maxsens_samples, device)
    mm.gpu_mem_report(f"after loading {model_name}")

    # ---- Process ONE XAI method at a time ----
    for method in METHOD_NAMES:
        explain_func = explain_funcs[method]
        print(f"\n  Method: {method} ({model_name})")

        metric = MaxSensitivity(**MAXSENS_KWARGS)

        n_images = x_batch_all.shape[0]
        pbar = tqdm(range(n_images), desc=f"MaxSensitivity [{model_name}/{method}]")

        for img_idx in pbar:
            if already_done(checkpoint_df, model_name, method, img_idx):
                continue  # already computed in a previous (crashed) run

            np.random.seed(SEED)
            random.seed(SEED)
            torch.manual_seed(SEED)

            x_single = x_batch_all[img_idx:img_idx + 1]
            y_single = y_batch_all[img_idx:img_idx + 1]
            a_single = None

            try:
                a_single = explain_func(model, x_single, y_single)

                score = metric(
                    model=model,
                    x_batch=x_single,
                    y_batch=y_single,
                    a_batch=a_single,
                    device=device,
                    explain_func=explain_func,
                )[0]

                row = {
                    "model": model_name,
                    "method": method,
                    "image_index": img_idx,
                    "max_sensitivity": float(score),
                }
                append_checkpoint_row(row)
                checkpoint_df = pd.concat([checkpoint_df, pd.DataFrame([row])], ignore_index=True)

            except Exception as e:
                print(f"    Error on {model_name}/{method}/image {img_idx}: {e}")

            finally:
                # Clear Python/PyTorch memory after EACH image 
                a_single = None
                x_single = None
                y_single = None
                mm.clear_memory()
                if img_idx % 5 == 0:
                    mm.log_rss(f"{model_name}/{method}/image {img_idx}")

        # Clear the MaxSensitivity object after EACH method 
        del metric
        mm.clear_memory()


        method_df = checkpoint_df[(checkpoint_df["model"] == model_name) & (checkpoint_df["method"] == method)]
        method_summary_path = os.path.join(RESULTS_DIR, f"maxsens_{model_name}_{method}_summary.csv")
        method_df.to_csv(method_summary_path, index=False)
        print(f"  Saved {len(method_df)} results for {model_name}/{method} -> {method_summary_path}")

        mm.gpu_mem_report(f"after {model_name}/{method}")

    del x_batch_all, y_batch_all

    mm.unload_model(model=model, engine=engine, engine_cache=_engine_cache)
    mm.gpu_mem_report(f"after unloading {model_name}")

print("\nAll models and methods complete.")


**Final summary and plot**

In [ ]:
full_maxsens_df = load_checkpoint()
display(full_maxsens_df)

maxsens_summary = (
    full_maxsens_df
    .groupby(['model', 'method'])['max_sensitivity']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean': 'mean_max_sensitivity', 'std': 'std_max_sensitivity', 'count': 'n_images'})
)
maxsens_summary['mean_max_sensitivity'] = maxsens_summary['mean_max_sensitivity'].round(4)
maxsens_summary['std_max_sensitivity'] = maxsens_summary['std_max_sensitivity'].round(4)
maxsens_summary = maxsens_summary.sort_values(['model', 'mean_max_sensitivity'])

print("\n=== MaxSensitivity summary -- mean score per model/method ===")
print("    (lower mean_max_sensitivity = explanation changes less under small input")
print("     perturbations = more robust explanation)\n")
display(maxsens_summary)

now = datetime.now().strftime("%Y-%m-%d")
full_maxsens_df.to_csv(os.path.join(RESULTS_DIR, f'maxsens_full_{now}.csv'), index=False)
maxsens_summary.to_csv(os.path.join(RESULTS_DIR, f'maxsens_summary_{now}.csv'), index=False)
print(f"\nSaved per-image results to maxsens_full_{now}.csv")
print(f"Saved summary to maxsens_summary_{now}.csv")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
pivot = maxsens_summary.pivot(index='method', columns='model', values='mean_max_sensitivity')
pivot.plot(kind='bar', ax=plt.gca())
plt.yscale("log")
plt.ylabel("Mean MaxSensitivity (log scale, lower = more robust)")
plt.title("MaxSensitivity Comparison by Model and Method")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
